# Notebook 2: Additional METABRIC analyses (comparators, original panel, normalisation, class weights)
**Paper:** SHAP-Guided Selection of Gene Panel with Repeated Nested Cross-Validation and PAM50-Exclusion Benchmarking for Interpretable Breast Cancer Subtype Classification from RNA-Seq, with External Validation on METABRIC
**Authors:** Neetu Ahlawat, Vishal Verma

Each section is independent and saves its results to `OUT_DIR`.

| Section | Analysis | CPU time (approx.) |
|---|---|---|
| 1 | Data loading (TCGA-BRCA and METABRIC) | 2 min |
| 2 | All seven comparator panels on METABRIC (Gain-300, L1-300, SHAP-300 without PAM50, etc.) | 45 min |
| 3 | The original 411-tumour training panel on METABRIC | 12 min |
| 4 | Single-sample (rank-based) normalisation instead of per-cohort z-scores (Section 4.8) | 2 min |
| 5 | Class-weighted training (HER2-enriched recall) | 3 min |

**Data:** upload `HiSeqV2`, `TCGA.BRCA.sampleMap_BRCA_clinicalMatrix` and `brca_metabric.tar.gz` (or the extracted `brca_metabric` folder) to `/content/`.

Survival, pathway enrichment and the panel-size / tuning analysis are in Notebook 3.


In [ ]:
# ===== 0. Setup and config =====
import os, sys, json, time, glob, tarfile, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binomtest
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

CONTENT      = os.environ.get("REV_CONTENT", "/content")
TCGA_EXPR    = f"{CONTENT}/HiSeqV2"                                   # .gz also works
TCGA_CLIN    = f"{CONTENT}/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix"   # or BRCA_clinicalMatrix
METABRIC_DIR = f"{CONTENT}/brca_metabric"
OUT_DIR      = f"{CONTENT}/revision_results"
# To keep results if Colab disconnects, mount Drive and use a Drive folder instead:
# from google.colab import drive; drive.mount('/content/drive'); OUT_DIR = "/content/drive/MyDrive/revision_results"

SEED       = 42
PANEL_SIZE = int(os.environ.get("REV_K", 300))
K_LIST     = [int(k) for k in os.environ.get("REV_KLIST", "50,100,200,300,500,1000").split(",")]
N_REPEATS  = int(os.environ.get("REV_REPEATS", 5))   # 5 repeats x 5 folds = 25 outer folds, as in the paper
N_BOOT     = 1000
RUN_NOPAM50_IN_CV   = True    # SHAP-300 without PAM50 inside every fold (one extra booster per fold)
RUN_ALLGENE_XGB_CV  = True    # XGBoost on all 20,530 genes inside every fold (slow on CPU)

def gpu_available():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False
USE_GPU_CV = gpu_available()  # GPU is used only in Section 6; Sections 2-3 use CPU to match the earlier runs exactly
print("xgboost", xgb.__version__, "| GPU for Section 6:", USE_GPU_CV)

CLASSES     = ["Basal", "Her2", "LumA", "LumB"]
CLASS_NAMES = ["Basal-like", "HER2-enriched", "Luminal A", "Luminal B"]
LABEL_MAP   = {c: i for i, c in enumerate(CLASSES)}
os.makedirs(OUT_DIR, exist_ok=True)
SUMMARY = {}   # every key number is collected here and printed in Section 9

In [ ]:
# ===== Helper functions (same settings as Table 2 of the paper) =====
ALIAS = {"GSG2": "HASPIN", "ORC6L": "ORC6", "KNTC2": "NDC80", "CDCA1": "NUF2", "KIAA0101": "PCLAF",
         "FAM64A": "PIMREG", "HN1": "JPT1", "KIAA1199": "CEMIP", "C1orf106": "INAVA", "MLF1IP": "CENPU",
         "CTSL2": "CTSV", "C17orf37": "MIEN1", "C10orf116": "ADIRF"}
PAM50 = ["ACTR3B","ANLN","BAG1","BCL2","BIRC5","BLVRA","CCNB1","CCNE1","CDC20","CDC6","CDH3","CENPF",
         "CEP55","CXXC5","EGFR","ERBB2","ESR1","EXO1","FGFR4","FOXA1","FOXC1","GPR160","GRB7","KIF2C",
         "KRT14","KRT17","KRT5","MAPT","MDM2","MELK","MIA","MKI67","MLPH","MMP11","MYBL2","MYC","NAT1",
         "NDC80","NUF2","ORC6","PGR","PHGDH","PTTG1","RRM2","SFRP1","SLC39A6","TMEM45B","TYMS","UBE2C","UBE2T"]

def harmonise(df):
    df = df.rename(columns=lambda g: ALIAS.get(str(g).strip(), str(g).strip()))
    if df.columns.duplicated().any():
        df = df.T.groupby(level=0).mean().T
    return df

def make_xgb(seed=SEED, gpu=False):
    kw = dict(objective="multi:softprob", n_estimators=300, max_depth=5, learning_rate=0.05,
              eval_metric="mlogloss", random_state=seed, n_jobs=-1)
    if gpu:
        kw.update(device="cuda", tree_method="hist")
    return XGBClassifier(**kw)

def make_models(seed=SEED, gpu=False, balanced=False):
    cw = "balanced" if balanced else None
    return {
        "XGBoost": make_xgb(seed, gpu),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1, class_weight=cw),
        "SVM": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=seed, class_weight=cw),
        "Logistic Regression": LogisticRegression(solver="lbfgs", max_iter=5000, class_weight=cw),
    }
MODELS = list(make_models())

def fit_model(name, model, X, y, balanced=False):
    if name == "XGBoost" and balanced:
        model.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
    else:
        model.fit(X, y)
    return model

def tree_shap_importance(booster_model, X):
    c = np.asarray(booster_model.get_booster().predict(xgb.DMatrix(np.asarray(X)), pred_contribs=True))
    if c.ndim == 2:
        c = c.reshape(X.shape[0], -1, X.shape[1] + 1)
    return np.abs(c[:, :, :-1]).mean(axis=(0, 1))

def gain_importance(booster_model, n_features):
    score = booster_model.get_booster().get_score(importance_type="gain")
    g = np.zeros(n_features)
    for key, val in score.items():
        g[int(key[1:])] = val
    return g

def top_k(primary, k, tiebreak=None):
    # rank by primary importance (descending); ties broken by tiebreak (descending), then by column order
    primary = np.nan_to_num(np.asarray(primary, float))
    if tiebreak is None:
        return np.argsort(-primary, kind="stable")[:k]
    tiebreak = np.nan_to_num(np.asarray(tiebreak, float))
    return np.lexsort((-tiebreak, -primary))[:k]

def l1_importance(X, y):
    ovr = OneVsRestClassifier(LogisticRegression(penalty="l1", C=0.5, solver="liblinear", max_iter=2000))
    ovr.fit(X, y)
    return np.mean([np.abs(est.coef_.ravel()) for est in ovr.estimators_], axis=0)

def scores(y, pred, proba=None):
    r = {"Accuracy (%)": accuracy_score(y, pred) * 100,
         "Balanced accuracy (%)": balanced_accuracy_score(y, pred) * 100,
         "Macro F1 (%)": f1_score(y, pred, average="macro") * 100,
         "Weighted F1 (%)": f1_score(y, pred, average="weighted") * 100}
    if proba is not None:
        try:
            r["Weighted ROC-AUC (%)"] = roc_auc_score(y, proba, multi_class="ovr", average="weighted", labels=[0, 1, 2, 3]) * 100
        except ValueError:
            r["Weighted ROC-AUC (%)"] = np.nan
    return r

def bootstrap_acc_ci(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); ok = (y == p).astype(float)
    b = np.array([ok[rng.integers(0, n, n)].mean() for _ in range(n_boot)]) * 100
    lo, hi = np.percentile(b, [2.5, 97.5])
    return f"{lo:.2f}-{hi:.2f}"

def mcnemar_exact(y, p1, p2):
    a = int(((p1 == y) & (p2 != y)).sum()); b = int(((p1 != y) & (p2 == y)).sum())
    return 1.0 if a + b == 0 else binomtest(a, a + b, 0.5).pvalue

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

def corrected_t(d, n_te_over_n_tr=0.25):
    d = np.asarray(d, float); N = len(d)
    v = d.var(ddof=1)
    if v == 0:
        return 0.0, 1.0
    t = d.mean() / np.sqrt((1 / N + n_te_over_n_tr) * v)
    return t, 2 * stats.t.sf(abs(t), N - 1)

In [ ]:
# ===== 1. Load TCGA-BRCA and METABRIC (same steps as the validation notebook) =====
if not os.path.exists(TCGA_EXPR) and os.path.exists(TCGA_EXPR + ".gz"):
    TCGA_EXPR += ".gz"
if not os.path.exists(TCGA_CLIN):
    alt = glob.glob(f"{CONTENT}/*BRCA_clinicalMatrix*")
    assert alt, "TCGA clinical matrix not found in /content"
    TCGA_CLIN = alt[0]

expr = pd.read_csv(TCGA_EXPR, sep="\t", index_col=0).T
expr.index = expr.index.str[:15]
expr = expr[~expr.index.duplicated(keep="first")]
expr = harmonise(expr.apply(pd.to_numeric, errors="coerce"))
clin = pd.read_csv(TCGA_CLIN, sep="\t", low_memory=False)
id_col = next(c for c in ["sampleID", "sample", "Sample ID", "SAMPLE_ID"] if c in clin.columns)
clin = clin.set_index(id_col); clin.index = clin.index.astype(str).str[:15]
clin = clin[~clin.index.duplicated(keep="first")]

NAME_FIX = {"Luminal A": "LumA", "Luminal B": "LumB", "Basal-like": "Basal", "HER2-enriched": "Her2",
            "Normal-like": "Normal", "LumA": "LumA", "LumB": "LumB", "Basal": "Basal", "Her2": "Her2", "Normal": "Normal"}
def labels_from(col):
    lab = clin[col].dropna().astype(str).str.strip().map(NAME_FIX)
    return lab[lab.isin(CLASSES)]

lab_t = labels_from("PAM50_mRNA_nature2012")
ids_t = expr.index.intersection(lab_t.index)
X_tcga = expr.loc[ids_t]
if X_tcga.isna().any().any():
    X_tcga = X_tcga.fillna(X_tcga.median())
y_tcga = lab_t.loc[ids_t].map(LABEL_MAP).astype(int).values
print("TCGA tumours (PAM50_mRNA_nature2012, four classes):", len(ids_t), "(paper: 514)")

# --- Check: would PAM50Call_RNAseq give a larger cohort?
if "PAM50Call_RNAseq" in clin.columns:
    lab_rs = labels_from("PAM50Call_RNAseq")
    ids_rs = expr.index.intersection(lab_rs.index)
    both = ids_t.intersection(ids_rs)
    agree = (lab_rs.loc[both] == lab_t.loc[both]).mean() * 100 if len(both) else np.nan
    print(f"PAM50Call_RNAseq: {len(ids_rs)} tumours with expression and a four-class label; "
          f"{len(both)} overlap with the 514; label agreement on overlap = {agree:.1f}%")
    print(lab_rs.loc[ids_rs].value_counts().to_dict())
    SUMMARY["PAM50Call_RNAseq tumours (4 classes)"] = len(ids_rs)
    SUMMARY["PAM50Call_RNAseq agreement with nature2012 (%)"] = round(float(agree), 1)
else:
    print("PAM50Call_RNAseq column not present in this clinical file")

# --- METABRIC
if not os.path.exists(os.path.join(METABRIC_DIR, "data_mrna_illumina_microarray.txt")):
    tars = sorted(glob.glob(f"{CONTENT}/brca_metabric*.tar.gz"), key=os.path.getsize, reverse=True)
    assert tars, "Upload brca_metabric.tar.gz (or the extracted folder) to /content first"
    with tarfile.open(tars[0]) as t:
        t.extractall(CONTENT)
mb = pd.read_csv(os.path.join(METABRIC_DIR, "data_mrna_illumina_microarray.txt"), sep="\t", comment="#", low_memory=False)
mb = mb.dropna(subset=["Hugo_Symbol"]).drop(columns=[c for c in ["Entrez_Gene_Id"] if c in mb.columns])
mb = harmonise(mb.set_index("Hugo_Symbol").T.apply(pd.to_numeric, errors="coerce"))
pat = pd.read_csv(os.path.join(METABRIC_DIR, "data_clinical_patient.txt"), sep="\t", comment="#", index_col="PATIENT_ID")
smp = pd.read_csv(os.path.join(METABRIC_DIR, "data_clinical_sample.txt"), sep="\t", comment="#")
s2p = dict(zip(smp["SAMPLE_ID"], smp["PATIENT_ID"]))
lab_m_all = pd.Series({s: pat.loc[s2p[s], "CLAUDIN_SUBTYPE"] if s2p.get(s) in pat.index else np.nan for s in mb.index})
lab_m = lab_m_all[lab_m_all.isin(CLASSES)]
X_mb = mb.loc[lab_m.index]
miss = X_mb.isna().mean(); X_mb = X_mb.loc[:, miss <= 0.20]; X_mb = X_mb.fillna(X_mb.median())
y_mb = lab_m.map(LABEL_MAP).astype(int).values
print("METABRIC tumours (four classes):", len(y_mb), "(previous run: 1608)")

common = sorted(set(X_tcga.columns) & set(X_mb.columns))
pam50_common = [g for g in PAM50 if g in common]
Xt_all = pd.DataFrame(StandardScaler().fit_transform(X_tcga.values), index=X_tcga.index, columns=X_tcga.columns)
Xt = Xt_all[common]
Xm = pd.DataFrame(StandardScaler().fit_transform(X_mb[common].values), index=X_mb.index, columns=common)
print(f"Common genes: {len(common)} (previous run: 16907) | PAM50 in both: {len(pam50_common)}/50")
majority = pd.Series(y_mb).value_counts().max() / len(y_mb) * 100

In [ ]:
# ===== 2. All seven comparators on METABRIC (adds Gain-300, L1-300, SHAP-300 without PAM50) =====
def cached_importance(name, fn):
    path = f"{OUT_DIR}/{name}.npy"
    if os.path.exists(path):
        print(f"[{name}] loaded from cache"); return np.load(path)
    t0 = time.time(); v = fn(); np.save(path, v)
    print(f"[{name}] done in {(time.time() - t0) / 60:.1f} min"); return v

def booster_importances(X, y):
    b = make_xgb().fit(X.values, y)
    return np.vstack([tree_shap_importance(b, X.values), gain_importance(b, X.shape[1])])

imp_common = cached_importance("imp_shap_gain_common", lambda: booster_importances(Xt, y_tcga))
F_common, _ = f_classif(Xt.values, y_tcga)
l1_common = cached_importance("imp_l1_common", lambda: l1_importance(Xt.values, y_tcga))
nopam = [g for g in common if g not in PAM50]
imp_nopam = cached_importance("imp_shap_nopam50_common", lambda: booster_importances(Xt[nopam], y_tcga))

cols = np.array(common)
panels_ext = {
    "SHAP-300":               list(cols[top_k(imp_common[0], PANEL_SIZE)]),
    "Gain-300":               list(cols[top_k(imp_common[1], PANEL_SIZE, F_common)]),
    "ANOVA-300":              list(cols[top_k(F_common, PANEL_SIZE)]),
    "L1-300":                 list(cols[top_k(l1_common, PANEL_SIZE, F_common)]),
    "SHAP-300 without PAM50": list(np.array(nopam)[top_k(imp_nopam[0], PANEL_SIZE)]),
    "PAM50":                  pam50_common,
    "All shared genes":       common,
}
prev = f"{CONTENT}/metabric_validation/panel_shap300_common.json"
if os.path.exists(prev):
    same = len(set(json.load(open(prev))) & set(panels_ext["SHAP-300"]))
    print(f"Overlap with the SHAP-300 panel of the earlier METABRIC run: {same}/{PANEL_SIZE} (should be {PANEL_SIZE})")

rows, preds_ext = [], {}
for pname, genes in panels_ext.items():
    for mname, model in make_models().items():
        fit_model(mname, model, Xt[genes].values, y_tcga)
        pr = model.predict_proba(Xm[genes].values); p = pr.argmax(1)
        preds_ext[(pname, mname)] = p
        rows.append({"Feature set": pname, "Model": mname, "n genes": len(genes), **scores(y_mb, p, pr),
                     "Accuracy 95% CI": bootstrap_acc_ci(y_mb, p)})
    print("done:", pname)
rng = np.random.default_rng(SEED); rand = {m: [] for m in MODELS}
for d in range(10):
    genes = list(rng.choice(common, PANEL_SIZE, replace=False))
    for mname, model in make_models(SEED + d).items():
        fit_model(mname, model, Xt[genes].values, y_tcga); pr = model.predict_proba(Xm[genes].values)
        rand[mname].append(scores(y_mb, pr.argmax(1), pr))
for mname in MODELS:
    dfr = pd.DataFrame(rand[mname])
    rows.append({"Feature set": "Random-300 (mean of 10)", "Model": mname, "n genes": PANEL_SIZE, **dfr.mean().to_dict()})
ext = pd.DataFrame(rows)
display(ext.round(2))

mc_rows = []
others = [p for p in panels_ext if p != "SHAP-300"]
for mname in MODELS:
    base = preds_ext[("SHAP-300", mname)]
    raw = [mcnemar_exact(y_mb, base, preds_ext[(o, mname)]) for o in others]
    for o, pr_, pa in zip(others, raw, holm(raw)):
        diff = (accuracy_score(y_mb, base) - accuracy_score(y_mb, preds_ext[(o, mname)])) * 100
        mc_rows.append({"Model": mname, "SHAP-300 minus": o, "Accuracy difference (points)": round(diff, 2),
                        "McNemar p": pr_, "Holm-adjusted p (6 comparisons)": pa})
mcn_ext = pd.DataFrame(mc_rows); display(mcn_ext.round(4))
ext.round(2).to_csv(f"{OUT_DIR}/sec2_metabric_all_comparators.csv", index=False)
mcn_ext.to_csv(f"{OUT_DIR}/sec2_mcnemar_all_comparators.csv", index=False)
SUMMARY["Sec2 METABRIC accuracy by feature set"] = ext.pivot(index="Feature set", columns="Model", values="Accuracy (%)").round(2).to_dict()
SUMMARY["Sec2 McNemar (Holm over 6)"] = mcn_ext.round(4).to_dict("records")

In [ ]:
# ===== 3. The ORIGINAL training-set panel (411 tumours, all 20,530 genes) on METABRIC =====
idx = np.arange(len(y_tcga))
tr, te = train_test_split(idx, test_size=0.2, stratify=y_tcga, random_state=SEED)
print("Training class counts:", np.bincount(y_tcga[tr]), "(paper: Basal 78, HER2 46, LumA 185, LumB 102)")
sc = StandardScaler().fit(X_tcga.values[tr])
Xtr_all = pd.DataFrame(sc.transform(X_tcga.values[tr]), columns=X_tcga.columns)
Xte_all = pd.DataFrame(sc.transform(X_tcga.values[te]), columns=X_tcga.columns)
imp411 = cached_importance("imp_shap_gain_train411_allgenes", lambda: booster_importances(Xtr_all, y_tcga[tr]))
panel411 = list(np.array(X_tcga.columns)[top_k(imp411[0], PANEL_SIZE)])
REV = {v: k for k, v in ALIAS.items()}
paper_top20 = ["TRIM29","FOXA1","MLPH","GRB7","FGFR4","CEP55","GSG2","CDC45","FAM171A1","RAB27B",
               "BCAN","TMEM86A","FOXM1","LOC150622","PNMT","GTSE1","DLGAP5","MCM10","CDCA5","PERP"]
ours20 = [REV.get(g, g) for g in panel411[:20]]
print("Top-20 now  :", ours20)
print("Top-20 paper:", paper_top20)
print(f"Reproduction check: {len(set(ours20) & set(paper_top20))}/20 of the paper's top-20 genes recovered")
json.dump(panel411, open(f"{OUT_DIR}/panel_train411_allgenes.json", "w"))

# held-out reproduction check (should match Table 3 if the split is the same)
ho = []
for mname, model in make_models().items():
    fit_model(mname, model, Xtr_all[panel411].values, y_tcga[tr])
    ho.append({"Model": mname, "Held-out accuracy now (%)": accuracy_score(y_tcga[te], model.predict(Xte_all[panel411].values)) * 100})
ho = pd.DataFrame(ho); ho["Paper Table 3 (%)"] = [87.38, 85.44, 88.35, 86.41]; display(ho.round(2))

matched411 = [g for g in panel411 if g in common]
print(f"Original panel genes measured in METABRIC: {len(matched411)}/{PANEL_SIZE}")
Xtr_c = Xtr_all[matched411].values
rows3 = []
for mname, model in make_models().items():
    fit_model(mname, model, Xtr_c, y_tcga[tr])
    pr = model.predict_proba(Xm[matched411].values); p = pr.argmax(1)
    rows3.append({"Model": mname, **scores(y_mb, p, pr), "Accuracy 95% CI": bootstrap_acc_ci(y_mb, p)})
sec3 = pd.DataFrame(rows3); display(sec3.round(2))
sec3.round(2).to_csv(f"{OUT_DIR}/sec3_original_panel_on_metabric.csv", index=False)
SUMMARY["Sec3 original panel: genes matched"] = len(matched411)
SUMMARY["Sec3 top-20 reproduced"] = len(set(ours20) & set(paper_top20))
SUMMARY["Sec3 held-out reproduction"] = ho.round(2).to_dict("records")
SUMMARY["Sec3 original panel on METABRIC"] = sec3.round(2).to_dict("records")

In [ ]:
# ===== 4. Single-sample normalisation: within-sample percentile ranks (no cohort statistics used) =====
def within_sample_rank(df):
    return df.rank(axis=1, pct=True)
Rt = within_sample_rank(X_tcga[common]); Rm = within_sample_rank(X_mb[common])
genes = panels_ext["SHAP-300"]
rows4 = []
for mname, model in make_models().items():
    fit_model(mname, model, Rt[genes].values, y_tcga)
    pr = model.predict_proba(Rm[genes].values); p = pr.argmax(1)
    z_acc = accuracy_score(y_mb, preds_ext[("SHAP-300", mname)]) * 100
    rows4.append({"Model": mname, **scores(y_mb, p, pr), "Accuracy 95% CI": bootstrap_acc_ci(y_mb, p),
                  "Per-cohort z-score accuracy (%)": z_acc})
sec4 = pd.DataFrame(rows4); display(sec4.round(2))
sec4.round(2).to_csv(f"{OUT_DIR}/sec4_rank_normalisation.csv", index=False)
SUMMARY["Sec4 rank-normalised SHAP-300 on METABRIC"] = sec4.round(2).to_dict("records")

In [ ]:
# ===== 5. Class-weighted training (does HER2-enriched recall improve?) =====
rows5 = []
for mname, model in make_models(balanced=True).items():
    fit_model(mname, model, Xt[genes].values, y_tcga, balanced=True)
    pr = model.predict_proba(Xm[genes].values); p = pr.argmax(1)
    _, rec_w, _, _ = precision_recall_fscore_support(y_mb, p, labels=[0, 1, 2, 3], zero_division=0)
    _, rec_u, _, _ = precision_recall_fscore_support(y_mb, preds_ext[("SHAP-300", mname)], labels=[0, 1, 2, 3], zero_division=0)
    rows5.append({"Model": mname, **scores(y_mb, p, pr),
                  "HER2 recall weighted": rec_w[1], "HER2 recall unweighted": rec_u[1],
                  "LumB recall weighted": rec_w[3], "LumB recall unweighted": rec_u[3],
                  "McNemar p vs unweighted": mcnemar_exact(y_mb, p, preds_ext[("SHAP-300", mname)])})
sec5 = pd.DataFrame(rows5); display(sec5.round(3))
sec5.round(4).to_csv(f"{OUT_DIR}/sec5_class_weights.csv", index=False)
SUMMARY["Sec5 class-weighted SHAP-300"] = sec5.round(3).to_dict("records")